In [5]:
# Install required libraries
!pip install google-cloud-bigquery pandas db-dtypes --quiet

In [6]:
# Import BigQuery client and set configuration variables
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-871084f9eb9e"
DATASET_ID = "emergency_response"
RAW_TABLE = "emergency_calls_raw"
MODEL_NAME = "response_time_model"
LOCATION = "US"

client = bigquery.Client(project=PROJECT_ID)

In [7]:
# Create BigQuery dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
client.create_dataset(dataset_ref, exists_ok=True)

print(f"Dataset {DATASET_ID} ready")

Dataset emergency_response ready


In [8]:
# Load emergency response data into BigQuery
load_job = client.load_table_from_uri(
    "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv",
    f"{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}",
    job_config=bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        autodetect=True,
        skip_leading_rows=1,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
)

load_job.result()
print("Data loaded")

Data loaded


In [9]:
# Preview the raw data
df = client.query(f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}` LIMIT 5").to_dataframe()
df

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37


In [12]:
#Train the model
train_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`
OPTIONS(
  model_type='linear_reg',
  input_label_cols=['response_time']
) AS
SELECT
  call_type,
  location,
  weather_condition,
  day_of_week,
  time_of_day,
  traffic_level,
  distance_to_station,
  units_available,
  response_time
FROM `{PROJECT_ID}.{DATASET_ID}.{RAW_TABLE}`
WHERE response_time IS NOT NULL
"""
client.query(train_sql).result()
print("Model created")

Model created


In [13]:
# Evaluate the model
eval_sql = f"""
SELECT *
FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`)
"""
eval_df = client.query(eval_sql).to_dataframe()
eval_df

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.761934,4.827846,0.015117,1.501836,0.831417,0.83146


In [14]:
# Predict using synthetic input data
predict_sql = f"""
SELECT *
FROM ML.PREDICT(
  MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_NAME}`,
  (
    SELECT
      'Fire' AS call_type,
      'Riverside' AS location,
      'Rainy' AS weather_condition,
      'Monday' AS day_of_week,
      1 AS time_of_day,
      'High' AS traffic_level,
      18.5 AS distance_to_station,
      12 AS units_available
  )
)
"""
pred_df = client.query(predict_sql).to_dataframe()
pred_df


,predicted_response_time,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,19.803368,Fire,Riverside,Rainy,Monday,1,High,18.5,12
